# Import Library

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import os

In [3]:
fake = Faker()
np.random.seed(42)  # reproducibility
random.seed(42)

# Load Dataset

In [2]:
os.chdir('..') if 'notebooks' in os.getcwd() else None


In [5]:
os.chdir('..')  # naik ke root folder
print(os.getcwd())  # verifikasi lokasi sekarang

d:\Project Portofolio\Data Analyst, Scientiest, ML, DL, AI\Retail Shrinkage & Loss Prevention Analytics\retail_shrinkage_analytics


In [6]:
transactions = pd.read_csv('raw_data/transaction_data.csv')
products     = pd.read_csv('raw_data/product.csv')

print("✅ Data loaded!")
print(f"   Transactions : {len(transactions):,} rows")
print(f"   Products     : {len(products):,} rows")

✅ Data loaded!
   Transactions : 2,595,732 rows
   Products     : 92,353 rows


In [7]:
stores       = transactions['STORE_ID'].unique()
product_ids  = transactions['PRODUCT_ID'].unique()
days         = transactions['DAY'].unique()

avg_sales    = transactions['SALES_VALUE'].mean()
std_sales    = transactions['SALES_VALUE'].std()

print(f"\n📊 Referensi statistik:")
print(f"   Unique stores   : {len(stores)}")
print(f"   Unique products : {len(product_ids):,}")
print(f"   Avg sales value : ${avg_sales:.2f}")
print(f"   Std sales value : ${std_sales:.2f}")


📊 Referensi statistik:
   Unique stores   : 582
   Unique products : 92,339
   Avg sales value : $3.10
   Std sales value : $4.18


# Shrinkage Simulation

## Simulasi Tipe 1 — Void Abuse

In [8]:
def simulate_void_abuse(transactions, n_fraudulent_stores=10, voids_per_store=50):
    """
    Simulasi kasir yang melakukan void berulang pada produk mahal.
    
    Logic:
    - Pilih beberapa store sebagai 'fraudulent store'
    - Inject transaksi void (SALES_VALUE negatif) dengan frekuensi abnormal
    - Concentrated di jam kerja tertentu (anomali waktu)
    """
    
    # Pilih store yang akan jadi 'fraudulent'
    fraudulent_stores = np.random.choice(stores, n_fraudulent_stores, replace=False)
    
    # Pilih produk mahal sebagai target void
    # Logic: kasir curang cenderung void produk bernilai tinggi
    expensive_products = transactions[
        transactions['SALES_VALUE'] > transactions['SALES_VALUE'].quantile(0.85)
    ]['PRODUCT_ID'].unique()
    
    void_records = []
    
    for store in fraudulent_stores:
        for _ in range(voids_per_store):
            void_records.append({
                'household_key' : 9999,  # placeholder = bukan pelanggan nyata
                'BASKET_ID'     : np.random.randint(99000000, 99999999),
                'DAY'           : np.random.choice(days),
                'PRODUCT_ID'    : np.random.choice(expensive_products),
                'QUANTITY'      : -1,  # negatif = void
                'SALES_VALUE'   : -abs(np.random.normal(avg_sales * 3, std_sales)),
                'STORE_ID'      : store,
                'RETAIL_DISC'   : 0.0,
                'TRANS_TIME'    : np.random.choice([800, 900, 1000, 1300, 1400]),
                'WEEK_NO'       : np.random.randint(1, 102),
                'COUPON_DISC'   : 0.0,
                'COUPON_MATCH_DISC' : 0.0,
                'SHRINKAGE_TYPE': 'VOID_ABUSE',  # label untuk evaluasi nanti
                'IS_FRAUD'      : 1
            })
    
    return pd.DataFrame(void_records)

Angka 0.85 adalah business judgment, bukan angka matematika murni. Di project nyata, angka ini bisa disesuaikan berdasarkan input dari tim Loss Prevention — "produk dengan harga berapa yang paling sering jadi target?"

In [9]:
# Test
void_df = simulate_void_abuse(transactions)
print(f"✅ Void Abuse simulasi   : {len(void_df):,} rows")
print(f"   Stores terlibat      : {void_df['STORE_ID'].nunique()}")
print(f"   Avg sales value      : ${void_df['SALES_VALUE'].mean():.2f}")
print(void_df.head(3))

✅ Void Abuse simulasi   : 500 rows
   Stores terlibat      : 10
   Avg sales value      : $-9.32
   household_key  BASKET_ID  DAY  PRODUCT_ID  QUANTITY  SALES_VALUE  STORE_ID  \
0           9999   99666490  469    12262732        -1   -16.897610       213   
1           9999   99154697  492       67254        -1   -11.004436       213   
2           9999   99283821  653     1024615        -1   -13.018699       213   

   RETAIL_DISC  TRANS_TIME  WEEK_NO  COUPON_DISC  COUPON_MATCH_DISC  \
0          0.0        1300        9          0.0                0.0   
1          0.0        1400        7          0.0                0.0   
2          0.0         900        5          0.0                0.0   

  SHRINKAGE_TYPE  IS_FRAUD  
0     VOID_ABUSE         1  
1     VOID_ABUSE         1  
2     VOID_ABUSE         1  


In [10]:
void_df

,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,SHRINKAGE_TYPE,IS_FRAUD
0,9999,99666490,469,12262732,-1,-16.897610,213,0.0,1300,9,0.0,0.0,VOID_ABUSE,1
1,9999,99154697,492,67254,-1,-11.004436,213,0.0,1400,7,0.0,0.0,VOID_ABUSE,1
2,9999,99283821,653,1024615,-1,-13.018699,213,0.0,900,5,0.0,0.0,VOID_ABUSE,1
3,9999,99612380,165,9835960,-1,-8.334949,213,0.0,1400,65,0.0,0.0,VOID_ABUSE,1
4,9999,99238067,657,17383388,-1,-6.307629,213,0.0,1400,54,0.0,0.0,VOID_ABUSE,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,9999,99853725,3,12524164,-1,-15.871584,2980,0.0,1400,8,0.0,0.0,VOID_ABUSE,1
496,9999,99863659,493,888625,-1,-5.804730,2980,0.0,1000,77,0.0,0.0,VOID_ABUSE,1
497,9999,99571441,523,878006,-1,-10.997070,2980,0.0,1300,70,0.0,0.0,VOID_ABUSE,1
498,9999,99028328,272,12350549,-1,-2.628565,2980,0.0,800,18,0.0,0.0,VOID_ABUSE,1


## Simulasi Tipe 2 — Sweep Fraud

In [11]:
def simulate_sweep_fraud(transactions, n_incidents=200):
    """
    Simulasi transaksi mencurigakan di luar jam operasional normal.
    
    Logic:
    - Transaksi terjadi di jam tidak wajar (tengah malam / subuh)
    - Nilai transaksi jauh di atas rata-rata (sweep = ambil banyak sekaligus)
    - Tidak terkait household nyata
    """
    
    sweep_records = []
    
    for _ in range(n_incidents):
        # Jam tidak wajar: 0000-0500 atau 2200-2359
        suspicious_times = list(range(0, 500, 30)) + list(range(2200, 2400, 30))
        
        sweep_records.append({
            'household_key' : 9999,
            'BASKET_ID'     : np.random.randint(88000000, 88999999),
            'DAY'           : np.random.choice(days),
            'PRODUCT_ID'    : np.random.choice(product_ids),
            'QUANTITY'      : np.random.randint(5, 20),  # quantity besar sekaligus
            'SALES_VALUE'   : abs(np.random.normal(avg_sales * 8, std_sales * 2)),
            'STORE_ID'      : np.random.choice(stores),
            'RETAIL_DISC'   : 0.0,
            'TRANS_TIME'    : np.random.choice(suspicious_times),
            'WEEK_NO'       : np.random.randint(1, 102),
            'COUPON_DISC'   : 0.0,
            'COUPON_MATCH_DISC' : 0.0,
            'SHRINKAGE_TYPE': 'SWEEP_FRAUD',
            'IS_FRAUD'      : 1
        })
    
    return pd.DataFrame(sweep_records)


In [12]:

sweep_df = simulate_sweep_fraud(transactions)
print(f"✅ Sweep Fraud simulasi  : {len(sweep_df):,} rows")
print(f"   Avg sales value      : ${sweep_df['SALES_VALUE'].mean():.2f}")
print(f"   Avg quantity         : {sweep_df['QUANTITY'].mean():.1f} units")

✅ Sweep Fraud simulasi  : 200 rows
   Avg sales value      : $25.76
   Avg quantity         : 11.9 units


## Simulasi Tipe 3 & 4

In [13]:
def simulate_coupon_fraud(transactions, n_incidents=300):
    """
    Kupon di-redeem tanpa pembelian valid.
    COUPON_DISC ada tapi SALES_VALUE = 0 dan QUANTITY mencurigakan.
    """
    coupon_records = []
    
    for _ in range(n_incidents):
        coupon_records.append({
            'household_key' : 9999,
            'BASKET_ID'     : np.random.randint(77000000, 77999999),
            'DAY'           : np.random.choice(days),
            'PRODUCT_ID'    : np.random.choice(product_ids),
            'QUANTITY'      : np.random.randint(1, 5),
            'SALES_VALUE'   : 0.0,
            'STORE_ID'      : np.random.choice(stores),
            'RETAIL_DISC'   : 0.0,
            'TRANS_TIME'    : np.random.randint(800, 2100),
            'WEEK_NO'       : np.random.randint(1, 102),
            'COUPON_DISC'   : -abs(np.random.uniform(1.0, 5.0)),  # diskon besar
            'COUPON_MATCH_DISC' : 0.0,
            'SHRINKAGE_TYPE': 'COUPON_FRAUD',
            'IS_FRAUD'      : 1
        })
    
    return pd.DataFrame(coupon_records)

In [14]:
def simulate_expired_waste(transactions, n_incidents=400):
    """
    Produk expired keluar tanpa nilai tercatat.
    Simulasi pencatatan waste — tidak ada di POS asli.
    """
    # Fokus ke kategori perishable
    perishable_depts = ['MEAT', 'PRODUCE', 'DAIRY', 'FROZEN FOODS', 'PASTRY']
    perishable_products = products[
        products['DEPARTMENT'].isin(perishable_depts)
    ]['PRODUCT_ID'].unique()
    
    # Fallback kalau produk perishable tidak ditemukan
    if len(perishable_products) == 0:
        perishable_products = product_ids
    
    waste_records = []
    
    for _ in range(n_incidents):
        waste_records.append({
            'household_key' : 8888,  # kode internal = bukan pelanggan
            'BASKET_ID'     : np.random.randint(66000000, 66999999),
            'DAY'           : np.random.choice(days),
            'PRODUCT_ID'    : np.random.choice(perishable_products),
            'QUANTITY'      : np.random.randint(1, 10),
            'SALES_VALUE'   : 0.0,
            'STORE_ID'      : np.random.choice(stores),
            'RETAIL_DISC'   : 0.0,
            'TRANS_TIME'    : np.random.choice([600, 700, 800, 2000, 2100]),
            'WEEK_NO'       : np.random.randint(1, 102),
            'COUPON_DISC'   : 0.0,
            'COUPON_MATCH_DISC' : 0.0,
            'SHRINKAGE_TYPE': 'EXPIRED_WASTE',
            'IS_FRAUD'      : 1
        })
    
    return pd.DataFrame(waste_records)

In [15]:
coupon_df = simulate_coupon_fraud(transactions)
waste_df  = simulate_expired_waste(transactions)

print(f"✅ Coupon Fraud simulasi : {len(coupon_df):,} rows")
print(f"✅ Expired Waste simulasi: {len(waste_df):,} rows")

✅ Coupon Fraud simulasi : 300 rows
✅ Expired Waste simulasi: 400 rows


# Final Dataset Summary

In [16]:
# Tambahkan label ke data asli (bukan fraud)
transactions['SHRINKAGE_TYPE'] = 'NORMAL'
transactions['IS_FRAUD']       = 0

In [17]:
# Gabungkan semua
full_dataset = pd.concat([
    transactions,
    void_df,
    sweep_df,
    coupon_df,
    waste_df
], ignore_index=True)

In [18]:
# Summary
print("=" * 55)
print("FINAL DATASET SUMMARY")
print("=" * 55)
print(f"\nTotal rows      : {len(full_dataset):,}")
print(f"Normal          : {(full_dataset['IS_FRAUD']==0).sum():,}")
print(f"Fraud/Shrinkage : {(full_dataset['IS_FRAUD']==1).sum():,}")
print(f"\nBreakdown shrinkage:")
print(full_dataset[full_dataset['IS_FRAUD']==1]['SHRINKAGE_TYPE'].value_counts())

FINAL DATASET SUMMARY

Total rows      : 2,597,132
Normal          : 2,595,732
Fraud/Shrinkage : 1,400

Breakdown shrinkage:
SHRINKAGE_TYPE
VOID_ABUSE       500
EXPIRED_WASTE    400
COUPON_FRAUD     300
SWEEP_FRAUD      200
Name: count, dtype: int64


In [19]:
# Simpan ke folder data
full_dataset.to_csv('data/transactions_with_shrinkage.csv', index=False)
print(f"\n✅ Saved → data/transactions_with_shrinkage.csv")


✅ Saved → data/transactions_with_shrinkage.csv
